In [ ]:
import ee

# NOTA: "viirs-peru" es el ID del proyecto de Google Cloud vinculado a Earth Engine, el nombre fue ese pero es para todo EARTH ENGINE
# sirve como autenticación para CUALQUIER dataset de Earth Engine (WorldCover, SRTM,
# Sentinel-2, Open Buildings, etc.), no solo para VIIRS.

try:
    ee.Initialize(project="viirs-peru")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="viirs-peru")

print("Google Earth Engine listo")

In [ ]:
from pathlib import Path
import ee
import geemap
import geopandas as gpd
import pandas as pd
import json
import math
import time

def find_project_root(marker="requirements.txt"):
    path = Path.cwd()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"No se encontró '{marker}' subiendo desde {path}")

PROJECT_ROOT = find_project_root()
DATA = PROJECT_ROOT / "data"

geobase = gpd.read_file(DATA / "clean" / "staging" / "geobase_distrital.gpkg")
print("Distritos en geobase:", len(geobase))

In [ ]:
distritos_base = geobase[["UBIGEO", "geometry"]].copy()
distritos_base["geometry"] = distritos_base.geometry.simplify(
    tolerance=0.005, preserve_topology=True
)
distritos_base = distritos_base[
    distritos_base.geometry.notna() & ~distritos_base.geometry.is_empty
].copy()

print("Distritos a procesar:", len(distritos_base))

## 3. Sentinel-2 NDVI — Contexto agrícola por distrito

**Fuente:** Sentinel-2 Level-2A (reflectancia de superficie) — `COPERNICUS/S2_SR_HARMONIZED` (Google Earth Engine)
**¿Usa IA?** No. El NDVI es una fórmula aritmética fija aplicada píxel por píxel `(B8-B4)/(B8+B4)`, sin parámetros aprendidos — mismo bloque metodológico que SRTM (interferometría de radar) y JRC (reglas espectrales fijas).
**Unión:** por `UBIGEO`, usando el mismo `limite_distrital` (shapefile INEI 2025) ya cargado, convertido a `ee.FeatureCollection` en lotes de 40 distritos (mismo enfoque que SRTM y JRC).

### Qué haremos

1. Cargar el límite distrital (`UBIGEO` + geometría del polígono) ya disponible.
2. Filtrar la colección Sentinel-2 SR Harmonized al año 2021 (consistencia con JRC; a revisar si SIAF o el registro de anemia terminan fijándose en otro año de referencia).
3. Enmascarar nubes y sombras de nube usando la banda `QA60` (o `SCL`) antes de calcular cualquier estadístico.
4. Calcular NDVI por imagen: `NDVI = (B8 - B4) / (B8 + B4)`.
5. Construir, por distrito, un compuesto anual libre de nubes y extraer tres estadísticos (no uno solo, para no promediar temporadas distintas): valor máximo (pico de crecimiento), valor mínimo (temporada seca / rastrojo) y promedio anual.
6. Agregar por polígono distrital con `reduceRegion` (lote por lote, igual que SRTM/JRC, por el límite de payload de 10MB).
7. Calcular `ndvi_amplitud = ndvi_max - ndvi_min` — variable derivada que distingue vegetación natural permanente (amplitud baja, siempre verde) de agricultura estacional activa (amplitud alta, cambia drásticamente entre siembra y cosecha).
8. Unir el resultado a la base distrital por `ubigeo`.

### Qué obtendremos (variables)

| Variable | Qué mide |
|---|---|
| `ndvi_promedio_anual` | Nivel general de verdor del distrito en el año |
| `ndvi_max_temporada` | Pico de vegetación (cultivo en pleno desarrollo) |
| `ndvi_min_temporada` | Valle de vegetación (rastrojo / temporada seca) |
| `ndvi_amplitud` | Diferencia max−min: intensidad de la actividad agrícola estacional |

### Por qué es relevante para el objetivo de anemia infantil

El NDVI y su amplitud estacional funcionan como proxy del **contexto agrícola del hogar rural**, no de qué cultivo específico se siembra (eso lo resolverá más adelante la segmentación satelital fina). Se incorpora al modelo porque:

- Aproxima la **base económica y capacidad de autoconsumo** del hogar — un territorio con agricultura estacional intensa tiene una dinámica de ingresos y alimentación distinta a uno sin actividad agrícola visible.
- Captura un **patrón de trabajo estacional** (migración temporal a la chacra en siembra/cosecha) que puede afectar la adherencia al seguimiento de suplementación de hierro — algo que ningún registro administrativo mide hoy.
- Sirve como **variable de contexto territorial** para el Causal Forest: junto con dispersión de viviendas (Open Buildings), agua (JRC) y elevación/pendiente (SRTM), ayuda a explicar heterogeneidad en el retorno marginal del gasto, no solo dónde hay pobreza.
- Es una **aproximación gruesa de etapa 0**: más adelante, la segmentación semántica sobre Planet NICFI (etapa de visión por computadora) reemplazará/refinará esto con conteo real de parcelas activas a nivel de caserío.

**Limitación declarada:** el NDVI no distingue tipo de cultivo, pastizal de cultivo, ni valor nutricional — solo intensidad y estacionalidad de la actividad fotosintética. Esta limitación se documentará en la pantalla de metodología del aplicativo final.

In [ ]:
# -----------------------------------------------------------------------------
# Enmascarado de nubes usando SCL (Scene Classification Layer)
# -----------------------------------------------------------------------------
# Usamos SCL en vez de QA60 porque QA60 quedó vacía/no confiable en muchas
# escenas desde 2022 (cambio de línea base de procesamiento de ESA). SCL sigue
# vigente en todo el rango 2021-2025, dando un enmascarado consistente.
#
# Clases SCL excluidas: 3 = sombra de nube, 8 = nube prob. media,
# 9 = nube prob. alta, 10 = cirros delgados.
def enmascarar_nubes_s2(imagen):
    scl = imagen.select('SCL')
    mascara = (scl.neq(3)
               .And(scl.neq(8))
               .And(scl.neq(9))
               .And(scl.neq(10)))
    return imagen.updateMask(mascara).copyProperties(imagen, ['system:time_start'])


def calcular_ndvi(imagen):
    ndvi = imagen.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return imagen.addBands(ndvi)


def procesar_anio_lote(anio, fc_lote):
    fecha_inicio = f"{anio}-01-01"
    fecha_fin = f"{anio}-12-31"

    coleccion = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                 .filterDate(fecha_inicio, fecha_fin)
                 .filterBounds(fc_lote)
                 .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80))
                 .map(enmascarar_nubes_s2)
                 .map(calcular_ndvi)
                 .select('NDVI'))

    reductor_combinado = (ee.Reducer.mean()
                           .combine(ee.Reducer.max(), sharedInputs=True)
                           .combine(ee.Reducer.min(), sharedInputs=True))

    imagen_compuesta = coleccion.reduce(reductor_combinado)

    resultado = imagen_compuesta.reduceRegions(
        collection=fc_lote,
        reducer=ee.Reducer.mean(),
        scale=30,
        tileScale=4
    )
    return resultado

## Descargar

In [ ]:
TAMANIO_LOTE = 40
ANIOS = list(range(2021, 2026))  # 2021, 2022, 2023, 2024, 2025

n_distritos = len(distritos_base)
n_lotes = math.ceil(n_distritos / TAMANIO_LOTE)

resultados_totales = []

for anio in ANIOS:
    print(f"Procesando año {anio}...")
    for i in range(n_lotes):
        inicio = i * TAMANIO_LOTE
        fin = inicio + TAMANIO_LOTE
        lote_gdf = distritos_base.iloc[inicio:fin]

        geojson_lote = json.loads(lote_gdf.to_json())
        fc_lote = ee.FeatureCollection(geojson_lote)
        fc_resultado = procesar_anio_lote(anio, fc_lote)

        try:
            datos_lote = fc_resultado.reduceColumns(
                ee.Reducer.toList(4),
                ['UBIGEO', 'NDVI_mean', 'NDVI_max', 'NDVI_min']
            ).get('list').getInfo()
        except Exception as e:
            print(f"  ⚠️  Error en lote {i+1} ({anio}): {e}")
            continue

        for fila in datos_lote:
            resultados_totales.append({
                'ubigeo': fila[0],
                'anio': anio,
                'ndvi_promedio_anual': fila[1],
                'ndvi_max_temporada': fila[2],
                'ndvi_min_temporada': fila[3],
            })

        print(f"  Lote {i+1}/{n_lotes} ({len(lote_gdf)} distritos) ✓")
        time.sleep(0.2)  # pequeño respiro para no saturar la API de GEE

print("Extracción completa.")

In [ ]:
df_ndvi = pd.DataFrame(resultados_totales)
df_ndvi['ndvi_amplitud'] = df_ndvi['ndvi_max_temporada'] - df_ndvi['ndvi_min_temporada']
df_ndvi = df_ndvi.sort_values(['ubigeo', 'anio']).reset_index(drop=True)

faltantes = df_ndvi[df_ndvi['ndvi_promedio_anual'].isna()]
if len(faltantes) > 0:
    print(f"⚠️  {len(faltantes)} filas con NDVI nulo — revisar estos distritos/años:")
    print(faltantes[['ubigeo', 'anio']])

print(df_ndvi.shape)
df_ndvi.head(10)

## guardar

In [ ]:
output_path = DATA / "clean" / "staging" / "sentinel2_ndvi_distrital_2021_2025.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

df_ndvi.to_csv(output_path, index=False)
print("Guardado en:", output_path)